# Notebook 6 — YOLO Feature Preparation for NLP

This notebook converts the final YOLOv5 detection output into image-level NLP-ready features.

**Input**
- `Outputs/YOLOv5_Detection/Final/yolov5_detections_full.csv`
- `Dataset/Processed/Captions/captions_processed.csv`

**Output**
- `Dataset/Processed/YOLO_Features/yolov5_image_features.csv`
- `Dataset/Processed/YOLO_Features/yolov5_detection_features.csv`

This notebook does not train the caption generator. It prepares object, confidence, and spatial features for the next model notebook.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Image_captioning")

print("Project exists:", PROJECT_DIR.exists())

if PROJECT_DIR.exists():
    print("\nProject contents:")
    for item in PROJECT_DIR.iterdir():
        print("[DIR]" if item.is_dir() else "[FILE]", item.name)

Project exists: True

Project contents:
[DIR] Dataset
[DIR] Notebooks
[DIR] Features
[DIR] Models
[DIR] Outputs


In [ ]:
# ============================================================
# CELL 1 — IMPORTS AND PATHS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import json

print("=" * 70)
print("NOTEBOOK 6 — YOLO FEATURE PREPARATION FOR NLP")
print("=" * 70)

PROJECT_DIR = Path("/content/drive/MyDrive/Image_captioning")

YOLO_DETECTIONS_FILE = (
    PROJECT_DIR / "Outputs" / "YOLOv5_Detection" / "Final"
    / "yolov5_detections_full.csv"
)

CAPTIONS_FILE = (
    PROJECT_DIR / "Dataset" / "Processed" / "Captions"
    / "captions_processed.csv"
)

OUTPUT_DIR = (
    PROJECT_DIR / "Dataset" / "Processed" / "YOLO_Features"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_FEATURES_FILE = OUTPUT_DIR / "yolov5_image_features.csv"
DETECTION_FEATURES_FILE = OUTPUT_DIR / "yolov5_detection_features.csv"

print("\nYOLO detections file:", YOLO_DETECTIONS_FILE)
print("Exists:", YOLO_DETECTIONS_FILE.exists())

print("\nCaptions file:", CAPTIONS_FILE)
print("Exists:", CAPTIONS_FILE.exists())

print("\nOutput directory:", OUTPUT_DIR)

assert YOLO_DETECTIONS_FILE.exists(), "YOLOv5 final detections file not found."
assert CAPTIONS_FILE.exists(), "Processed captions file not found."

print("\n✓ PATHS VERIFIED")


NOTEBOOK 6 — YOLO FEATURE PREPARATION FOR NLP

YOLO detections file: /content/drive/MyDrive/Image_captioning/Outputs/YOLOv5_Detection/Final/yolov5_detections_full.csv
Exists: True

Captions file: /content/drive/MyDrive/Image_captioning/Dataset/Processed/Captions/captions_processed.csv
Exists: True

Output directory: /content/drive/MyDrive/Image_captioning/Dataset/Processed/YOLO_Features

✓ PATHS VERIFIED


In [ ]:
# ============================================================
# CELL 2 — LOAD AND VERIFY YOLO DETECTIONS
# ============================================================

print("=" * 70)
print("LOADING FINAL YOLOv5 DETECTIONS")
print("=" * 70)

yolo_df = pd.read_csv(YOLO_DETECTIONS_FILE)

print("\nShape:", yolo_df.shape)
print("\nColumns:", list(yolo_df.columns))
print("\nUnique images:", yolo_df["image_id"].nunique())
print("\nTotal detection rows:", len(yolo_df))

print("\nMissing values:")
print(yolo_df.isna().sum())

display(yolo_df.head())

EXPECTED_IMAGES = 39874

assert yolo_df["image_id"].nunique() == EXPECTED_IMAGES, (
    f"Expected {EXPECTED_IMAGES} unique images."
)

print("\n✓ YOLO DETECTION DATA VERIFIED")


LOADING FINAL YOLOv5 DETECTIONS

Shape: (236748, 9)

Columns: ['image_id', 'image_path', 'object_name', 'confidence', 'xmin', 'ymin', 'xmax', 'ymax', 'objects_detected']

Unique images: 39874

Total detection rows: 236748

Missing values:
image_id              0
image_path            0
object_name         241
confidence          241
xmin                241
ymin                241
xmax                241
ymax                241
objects_detected      0
dtype: int64


,image_id,image_path,object_name,confidence,xmin,ymin,xmax,ymax,objects_detected
0,Flickr30K/1000092795.jpg,/content/Image_captioning_Local/Flickr30K/flic...,person,0.872264,160.006165,109.160004,259.443207,370.212219,1
1,Flickr30K/10002456.jpg,/content/Image_captioning_Local/Flickr30K/flic...,traffic light,0.545210,100.733315,160.662567,181.509979,264.724121,2
2,Flickr30K/10002456.jpg,/content/Image_captioning_Local/Flickr30K/flic...,traffic light,0.436365,244.372147,146.445236,372.893829,281.624512,2
3,Flickr30K/1000268201.jpg,/content/Image_captioning_Local/Flickr30K/flic...,person,0.895094,5.507481,150.834213,107.664742,339.458618,3
4,Flickr30K/1000268201.jpg,/content/Image_captioning_Local/Flickr30K/flic...,vase,0.430036,284.895966,234.392044,298.524323,277.204407,3



✓ YOLO DETECTION DATA VERIFIED


In [ ]:
# ============================================================
# CELL 3 — LOAD MASTER IMAGE MANIFEST
# ============================================================

print("=" * 70)
print("LOADING CAPTION MANIFEST")
print("=" * 70)

captions_df = pd.read_csv(CAPTIONS_FILE)

print("\nCaption file shape:", captions_df.shape)
print("Columns:", list(captions_df.columns))

image_manifest = (
    captions_df[
        ["image_id", "dataset", "image_name"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\nUnique images:", len(image_manifest))

print("\nDataset distribution:")
print(image_manifest["dataset"].value_counts())

assert len(image_manifest) == 39874, (
    "Expected exactly 39,874 unique images."
)

print("\n✓ CAPTION MANIFEST VERIFIED")


LOADING CAPTION MANIFEST

Caption file shape: (199369, 7)
Columns: ['image_id', 'dataset', 'image_name', 'caption', 'processed_caption', 'model_caption', 'caption_length']

Unique images: 39874

Dataset distribution:
dataset
Flickr30K    31783
Flickr8K      8091
Name: count, dtype: int64

✓ CAPTION MANIFEST VERIFIED


In [ ]:
# ============================================================
# CELL 4 — CLEAN YOLO DETECTION FEATURES
# ============================================================

print("=" * 70)
print("CLEANING YOLO DETECTION FEATURES")
print("=" * 70)

detection_df = yolo_df.copy()

detection_df["object_token"] = (
    detection_df["object_name"]
    .fillna("no_object")
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

numeric_columns = [
    "confidence",
    "xmin",
    "ymin",
    "xmax",
    "ymax",
    "objects_detected"
]

for column in numeric_columns:
    detection_df[column] = pd.to_numeric(
        detection_df[column],
        errors="coerce"
    )

zero_detection_mask = (
    detection_df["objects_detected"] == 0
)

detection_df.loc[
    zero_detection_mask,
    "confidence"
] = 0.0

coordinate_columns = [
    "xmin",
    "ymin",
    "xmax",
    "ymax"
]

detection_df[coordinate_columns] = (
    detection_df[coordinate_columns]
    .fillna(0.0)
)

print("\nMost common detected objects:")
print(
    detection_df["object_token"]
    .value_counts()
    .head(20)
)

print("\nZero-detection images:")
print(
    detection_df.loc[
        zero_detection_mask,
        "image_id"
    ].nunique()
)

print("\n✓ DETECTION VALUES CLEANED")


CLEANING YOLO DETECTION FEATURES

Most common detected objects:
object_token
person           151106
car                9201
chair              8948
handbag            5637
dog                5363
bottle             4099
cup                3324
bicycle            3323
backpack           3131
sports_ball        2471
umbrella           2198
book               1874
bowl               1728
motorcycle         1650
bird               1630
potted_plant       1540
traffic_light      1512
bench              1471
cell_phone         1383
dining_table       1375
Name: count, dtype: int64

Zero-detection images:
241

✓ DETECTION VALUES CLEANED


In [ ]:
# ============================================================
# CELL 5 — CREATE NORMALIZED SPATIAL FEATURES
# ============================================================

print("=" * 70)
print("CREATING NORMALIZED SPATIAL FEATURES")
print("=" * 70)

# NOTE:
# The original YOLO CSV does not contain original image width/height.
# For this prepared feature representation, per-image maximum xmax/ymax
# values are used as coordinate references.

image_dimensions = (
    detection_df
    .groupby("image_id")
    .agg(
        image_width_reference=("xmax", "max"),
        image_height_reference=("ymax", "max")
    )
    .reset_index()
)

detection_df = detection_df.merge(
    image_dimensions,
    on="image_id",
    how="left"
)

detection_df["image_width_reference"] = (
    detection_df["image_width_reference"]
    .replace(0, np.nan)
    .fillna(1.0)
)

detection_df["image_height_reference"] = (
    detection_df["image_height_reference"]
    .replace(0, np.nan)
    .fillna(1.0)
)

detection_df["box_width"] = (
    detection_df["xmax"]
    - detection_df["xmin"]
)

detection_df["box_height"] = (
    detection_df["ymax"]
    - detection_df["ymin"]
)

detection_df["center_x_norm"] = (
    (detection_df["xmin"] + detection_df["xmax"]) / 2
) / detection_df["image_width_reference"]

detection_df["center_y_norm"] = (
    (detection_df["ymin"] + detection_df["ymax"]) / 2
) / detection_df["image_height_reference"]

detection_df["box_width_norm"] = (
    detection_df["box_width"]
    / detection_df["image_width_reference"]
)

detection_df["box_height_norm"] = (
    detection_df["box_height"]
    / detection_df["image_height_reference"]
)

spatial_columns = [
    "center_x_norm",
    "center_y_norm",
    "box_width_norm",
    "box_height_norm"
]

for column in spatial_columns:
    detection_df[column] = (
        detection_df[column]
        .clip(lower=0.0, upper=1.0)
        .fillna(0.0)
    )

display(
    detection_df[
        [
            "image_id",
            "object_token",
            "confidence",
            "center_x_norm",
            "center_y_norm",
            "box_width_norm",
            "box_height_norm"
        ]
    ].head(10)
)

print("\n✓ NORMALIZED SPATIAL FEATURES CREATED")


CREATING NORMALIZED SPATIAL FEATURES


,image_id,object_token,confidence,center_x_norm,center_y_norm,box_width_norm,box_height_norm
0,Flickr30K/1000092795.jpg,person,0.872264,0.808365,0.647429,0.383271,0.705142
1,Flickr30K/10002456.jpg,traffic_light,0.545210,0.378450,0.755237,0.216621,0.369505
2,Flickr30K/10002456.jpg,traffic_light,0.436365,0.827670,0.760001,0.344660,0.479998
3,Flickr30K/1000268201.jpg,person,0.895094,0.151528,0.722169,0.273559,0.555662
4,Flickr30K/1000268201.jpg,vase,0.430036,0.781148,0.753548,0.036494,0.126120
5,Flickr30K/1000268201.jpg,potted_plant,0.292969,0.855440,0.564508,0.289120,0.409263
6,Flickr30K/1000344755.jpg,person,0.849371,0.871942,0.641618,0.256117,0.716765
7,Flickr30K/1000366164.jpg,person,0.915866,0.835731,0.555735,0.328538,0.875407
8,Flickr30K/1000366164.jpg,person,0.833118,0.294086,0.536931,0.583217,0.926138
9,Flickr30K/1000366164.jpg,bottle,0.525566,0.211663,0.778080,0.027902,0.107236



✓ NORMALIZED SPATIAL FEATURES CREATED


In [ ]:
# ============================================================
# CELL 6 — BUILD IMAGE-LEVEL FEATURE SEQUENCES
# ============================================================

print("=" * 70)
print("BUILDING IMAGE-LEVEL YOLO FEATURES")
print("=" * 70)


# ------------------------------------------------------------
# SORT DETECTIONS
# Highest confidence objects first
# ------------------------------------------------------------

sorted_detection_df = (
    detection_df
    .sort_values(
        by=["image_id", "confidence"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# BUILD IMAGE-LEVEL FEATURE RECORDS
# ------------------------------------------------------------

image_feature_records = []


for image_id, group in sorted_detection_df.groupby(
    "image_id",
    sort=False
):

    # Check whether this image has zero detections
    zero_detection = (
        group["objects_detected"].iloc[0] == 0
    )


    # --------------------------------------------------------
    # ZERO DETECTION IMAGE
    # --------------------------------------------------------

    if zero_detection:

        object_tokens = []

        confidences = []

        spatial_features = []

        detection_count = 0


    # --------------------------------------------------------
    # IMAGE WITH DETECTIONS
    # --------------------------------------------------------

    else:

        object_tokens = (
            group["object_token"]
            .astype(str)
            .tolist()
        )


        confidences = [

            round(
                float(value),
                6
            )

            for value in group[
                "confidence"
            ].tolist()

        ]


        spatial_features = []


        for _, row in group.iterrows():

            spatial_features.append(

                [

                    round(
                        float(
                            row["center_x_norm"]
                        ),
                        6
                    ),

                    round(
                        float(
                            row["center_y_norm"]
                        ),
                        6
                    ),

                    round(
                        float(
                            row["box_width_norm"]
                        ),
                        6
                    ),

                    round(
                        float(
                            row["box_height_norm"]
                        ),
                        6
                    )

                ]

            )


        detection_count = (
            len(object_tokens)
        )


    # --------------------------------------------------------
    # STORE IMAGE FEATURES
    # --------------------------------------------------------

    image_feature_records.append(

        {

            "image_id": image_id,

            "detected_object_count":
                detection_count,

            "object_tokens":
                json.dumps(object_tokens),

            "confidence_features":
                json.dumps(confidences),

            "spatial_features":
                json.dumps(spatial_features),

            "has_detection":
                int(
                    detection_count > 0
                )

        }

    )


# ------------------------------------------------------------
# CREATE DATAFRAME
# ------------------------------------------------------------

image_features_df = pd.DataFrame(
    image_feature_records
)


# ------------------------------------------------------------
# VERIFICATION
# ------------------------------------------------------------

print(
    "\nImage-level feature records:",
    len(image_features_df)
)


print(
    "\nImages with detections:",
    image_features_df[
        "has_detection"
    ].sum()
)


print(
    "\nImages with zero detections:",
    (
        image_features_df[
            "has_detection"
        ] == 0
    ).sum()
)


display(
    image_features_df.head()
)


assert len(
    image_features_df
) == 39874, (
    "Image-level feature count does not match "
    "the expected dataset size."
)


assert (
    image_features_df[
        "image_id"
    ].nunique()
    == 39874
), (
    "Unique image count does not match "
    "the expected dataset size."
)


print(
    "\n✓ IMAGE-LEVEL FEATURES CREATED SUCCESSFULLY"
)

BUILDING IMAGE-LEVEL YOLO FEATURES

Image-level feature records: 39874

Images with detections: 39633

Images with zero detections: 241


,image_id,detected_object_count,object_tokens,confidence_features,spatial_features,has_detection
0,Flickr30K/1000092795.jpg,1,"[""person""]",[0.872264],"[[0.808365, 0.647429, 0.383271, 0.705142]]",1
1,Flickr30K/10002456.jpg,2,"[""traffic_light"", ""traffic_light""]","[0.54521, 0.436365]","[[0.37845, 0.755237, 0.216621, 0.369505], [0.8...",1
2,Flickr30K/1000268201.jpg,3,"[""person"", ""vase"", ""potted_plant""]","[0.895094, 0.430036, 0.292969]","[[0.151528, 0.722169, 0.273559, 0.555662], [0....",1
3,Flickr30K/1000344755.jpg,1,"[""person""]",[0.849371],"[[0.871942, 0.641618, 0.256117, 0.716765]]",1
4,Flickr30K/1000366164.jpg,8,"[""person"", ""person"", ""bottle"", ""cup"", ""bottle""...","[0.915866, 0.833118, 0.525566, 0.451407, 0.448...","[[0.835731, 0.555735, 0.328538, 0.875407], [0....",1



✓ IMAGE-LEVEL FEATURES CREATED SUCCESSFULLY


In [ ]:
# ============================================================
# CELL 7 — VERIFY COMPLETE DATASET COVERAGE
# ============================================================

print("=" * 70)
print("VERIFYING COMPLETE IMAGE COVERAGE")
print("=" * 70)

final_image_features_df = image_manifest.merge(
    image_features_df,
    on="image_id",
    how="left"
)

missing_feature_images = (
    final_image_features_df["has_detection"]
    .isna()
    .sum()
)

print("\nTotal manifest images:", len(final_image_features_df))
print("Images missing YOLO feature records:", missing_feature_images)

print("\nDataset distribution:")
print(
    final_image_features_df
    .groupby("dataset")["image_id"]
    .nunique()
)

assert missing_feature_images == 0, (
    "Some images are missing YOLO feature records."
)

assert len(final_image_features_df) == 39874, (
    "Expected exactly 39,874 final image feature records."
)

print("\n✓ ALL 39,874 IMAGES HAVE YOLO FEATURE RECORDS")


VERIFYING COMPLETE IMAGE COVERAGE

Total manifest images: 39874
Images missing YOLO feature records: 0

Dataset distribution:
dataset
Flickr30K    31783
Flickr8K      8091
Name: image_id, dtype: int64

✓ ALL 39,874 IMAGES HAVE YOLO FEATURE RECORDS


In [ ]:
# ============================================================
# CELL 8 — SAVE BOTH FEATURE OUTPUTS
# ============================================================

print("=" * 70)
print("SAVING YOLO FEATURE OUTPUTS")
print("=" * 70)

detection_output_columns = [
    "image_id",
    "image_path",
    "object_name",
    "object_token",
    "confidence",
    "xmin",
    "ymin",
    "xmax",
    "ymax",
    "center_x_norm",
    "center_y_norm",
    "box_width_norm",
    "box_height_norm",
    "objects_detected"
]

final_detection_features_df = (
    sorted_detection_df[
        detection_output_columns
    ]
    .copy()
)

final_detection_features_df.to_csv(
    DETECTION_FEATURES_FILE,
    index=False
)

final_image_features_df.to_csv(
    IMAGE_FEATURES_FILE,
    index=False
)

print("\nDetection-level features saved:")
print(DETECTION_FEATURES_FILE)
print("Shape:", final_detection_features_df.shape)

print("\nImage-level features saved:")
print(IMAGE_FEATURES_FILE)
print("Shape:", final_image_features_df.shape)

print("\n✓ FEATURE FILES SAVED SUCCESSFULLY")


SAVING YOLO FEATURE OUTPUTS

Detection-level features saved:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/YOLO_Features/yolov5_detection_features.csv
Shape: (236748, 14)

Image-level features saved:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/YOLO_Features/yolov5_image_features.csv
Shape: (39874, 8)

✓ FEATURE FILES SAVED SUCCESSFULLY


In [ ]:
# ============================================================
# CELL 9 — FINAL VERIFICATION
# ============================================================

print("=" * 70)
print("NOTEBOOK 6 — FINAL VERIFICATION")
print("=" * 70)

verification_image_df = pd.read_csv(
    IMAGE_FEATURES_FILE
)

verification_detection_df = pd.read_csv(
    DETECTION_FEATURES_FILE
)

print("\nIMAGE-LEVEL FEATURES")
print("-" * 70)
print("Rows:", len(verification_image_df))
print(
    "Unique images:",
    verification_image_df["image_id"].nunique()
)
print(
    "Images with detections:",
    verification_image_df["has_detection"].sum()
)
print(
    "Images with zero detections:",
    (verification_image_df["has_detection"] == 0).sum()
)

print("\nDETECTION-LEVEL FEATURES")
print("-" * 70)
print("Rows:", len(verification_detection_df))
print(
    "Unique images:",
    verification_detection_df["image_id"].nunique()
)

assert len(verification_image_df) == 39874
assert verification_image_df["image_id"].nunique() == 39874
assert verification_detection_df["image_id"].nunique() == 39874
assert IMAGE_FEATURES_FILE.exists()
assert DETECTION_FEATURES_FILE.exists()

print("\n" + "=" * 70)
print("✓ NOTEBOOK 6 COMPLETE")
print("=" * 70)

print("\nFinal image-level NLP input:")
print(IMAGE_FEATURES_FILE)

print("\nFinal detection-level research output:")
print(DETECTION_FEATURES_FILE)

print("\nNext stage:")
print("Notebook 7 — Model 1 NLP Caption Generation Pipeline")


NOTEBOOK 6 — FINAL VERIFICATION

IMAGE-LEVEL FEATURES
----------------------------------------------------------------------
Rows: 39874
Unique images: 39874
Images with detections: 39633
Images with zero detections: 241

DETECTION-LEVEL FEATURES
----------------------------------------------------------------------
Rows: 236748
Unique images: 39874

✓ NOTEBOOK 6 COMPLETE

Final image-level NLP input:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/YOLO_Features/yolov5_image_features.csv

Final detection-level research output:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/YOLO_Features/yolov5_detection_features.csv

Next stage:
Notebook 7 — Model 1 NLP Caption Generation Pipeline
